In [ ]:
import os
import json
from datasets import load_dataset


def convert_split(split, name, out_dir):
    records = []

    kept = 0
    skipped_missing_fields = 0
    skipped_empty_answer = 0

    for ex in split:
        question = ex.get("question")
        sparql_dbpedia = ex.get("sparql_dbpedia18")
        sparql_wikidata = ex.get("sparql_wikidata")
        solved_answer = ex.get("solved_answer")

        # Require required fields
        if not question or not sparql_dbpedia or not sparql_wikidata:
            skipped_missing_fields += 1
            continue

        # Require a non-empty result set
        # (solved_answer is usually a list; skip [] / None)
        if not solved_answer:
            skipped_empty_answer += 1
            continue

        records.append(
            {
                "question": question.strip(),
                "sparql_dbpedia18": sparql_dbpedia.strip(),
                "sparql_wikidata": sparql_wikidata.strip(),
                "answer": solved_answer,  # keep as list
            }
        )
        kept += 1

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{name}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print(
        f"[{name}] kept={kept} | skipped_missing_fields={skipped_missing_fields} | skipped_empty_answer={skipped_empty_answer}"
    )
    print(f"Saved: {out_path}")


def main():
    print("Loading Orange/lc_quad2-sparqltotext ...")
    ds = load_dataset("Orange/lc_quad2-sparqltotext")
    print(ds)

    out_dir = "dataset"
    convert_split(ds["train"], "train", out_dir)
    convert_split(ds["validation"], "validation", out_dir)
    convert_split(ds["test"], "test", out_dir)


if __name__ == "__main__":
    main()

In [1]:
# rq1_qwen_sparql2text_encoder_eval_and_finetune.py
# Fine-tune Qwen on SPARQL -> Natural Text, then use the fine-tuned model
# as an encoder for retrieval (SPARQL embeddings).
#
# Fixes included:
#  - Filter examples with empty answer sets in load_examples (evaluation correctness)
#  - answer_jaccard: remove empty-empty => 1.0
#  - Added missing build_prompt_and_target()
#  - Avoid re-encoding corpus twice in RQ1

import os
import gc
import json
import random
import inspect
from typing import Any, Dict, List, Tuple

import numpy as np
import torch
from datasets import Dataset, load_dataset
from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from tqdm.auto import tqdm

from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)


# ============================================================
# Config
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
DATA_DIR = "dataset"
OUTPUT_DIR = "./qwen_sparql2text_encoder_rq1"

USE_FIELD = "sparql_dbpedia18"  # or "sparql_wikidata"
MAX_SEQ_LENGTH = 768

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
SEED = 42

EMBED_BATCH_SIZE = 8
QUESTION_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Answer-Jaccard threshold for semantic relevance
ANSWER_JACCARD_THRESHOLD = 0.8

TOPK = [1, 5, 10]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# Utils
# ============================================================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_json(path: str) -> List[Dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: Any, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return x / x.norm(dim=-1, keepdim=True).clamp(min=eps)


def cosine_sim_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    # expects already-normalized embeddings, but we normalize to be safe
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-12)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-12)
    return a_norm @ b_norm.T


# ============================================================
# Data loading (strict semantic evaluation)
# ============================================================

def safe_list(x: Any) -> List[str]:
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    s = str(x).strip()
    return [s] if s else []


def normalize_answer_item(x: str) -> str:
    # minimal normalization for matching
    return " ".join(str(x).strip().split())


def normalize_answer_list(answer: Any) -> List[str]:
    vals = safe_list(answer)
    return [normalize_answer_item(v) for v in vals if normalize_answer_item(v)]


def load_examples(path: str, use_field: str) -> List[Dict[str, Any]]:
    raw = load_json(path)
    items = []

    for i, ex in enumerate(raw):
        sparql = ex.get(use_field)
        question = ex.get("question")
        answers = normalize_answer_list(ex.get("answer"))

        if sparql is None or not str(sparql).strip():
            continue
        if question is None or not str(question).strip():
            continue

        # IMPORTANT FIX:
        # Filter out empty answer sets so ground truth doesn't treat empty-empty as relevant.
        if len(answers) == 0:
            continue

        items.append(
            {
                "id": ex.get("id", i),
                "sparql": str(sparql).strip(),
                "question": str(question).strip(),
                "answer": answers,
            }
        )

    return items


# ============================================================
# Fine-tuning data: SPARQL -> natural language
# ============================================================

def build_prompt_and_target(example: Dict[str, Any]) -> Tuple[str, str]:
    prompt = f"""You are an expert in knowledge graphs.

Convert the SPARQL query into a clear natural language question.

### Instruction
Convert the SPARQL query into a clear natural language question.

### SPARQL Query
{example['sparql']}

### Natural Text
"""
    target = example["question"]
    return prompt, target


def tokenize_supervised_example(example: Dict[str, Any], tokenizer, max_length: int) -> Dict[str, Any]:
    prompt, target = build_prompt_and_target(example)

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(target + tokenizer.eos_token, add_special_tokens=False)["input_ids"]

    input_ids_full = prompt_ids + target_ids
    input_ids = input_ids_full[:max_length]

    # attention mask
    attention_mask = [1] * len(input_ids)

    # labels: mask prompt tokens (-100), train only on target tokens
    labels_full = ([-100] * len(prompt_ids)) + target_ids
    labels = labels_full[:max_length]

    return {
        "id": example["id"],
        "sparql": example["sparql"],
        "question": example["question"],
        "answer": example["answer"],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


class SupervisedDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_len = max(len(f["input_ids"]) for f in features)

        pad_token_id = self.tokenizer.pad_token_id
        if pad_token_id is None:
            # should not happen if we set pad_token
            pad_token_id = self.tokenizer.eos_token_id

        input_ids, attention_mask, labels = [], [], []

        for f in features:
            pad_len = max_len - len(f["input_ids"])

            input_ids.append(f["input_ids"] + [pad_token_id] * pad_len)
            attention_mask.append(f["attention_mask"] + [0] * pad_len)
            labels.append(f["labels"] + [-100] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def build_tokenized_train_val_datasets(tokenizer):
    train_examples = load_examples(os.path.join(DATA_DIR, "train.json"), USE_FIELD)
    val_examples = load_examples(os.path.join(DATA_DIR, "validation.json"), USE_FIELD)

    train_tok = [tokenize_supervised_example(ex, tokenizer, MAX_SEQ_LENGTH) for ex in train_examples]
    val_tok = [tokenize_supervised_example(ex, tokenizer, MAX_SEQ_LENGTH) for ex in val_examples]

    train_ds = Dataset.from_list(train_tok)
    val_ds = Dataset.from_list(val_tok)
    return train_ds, val_ds


# ============================================================
# TrainingArguments (version-safe)
# ============================================================

def build_training_args() -> TrainingArguments:
    kwargs = dict(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=True,
        report_to="none",
    )

    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in sig.parameters:
        kwargs["evaluation_strategy"] = "steps"

    if "optim" in sig.parameters:
        kwargs["optim"] = "paged_adamw_8bit"

    return TrainingArguments(**kwargs)


# ============================================================
# QwenSPARQLEncoder: use last hidden state pooled over tokens
# ============================================================

class QwenSPARQLEncoder:
    def __init__(self, lora_path: str, device: str = DEVICE):
        self.device = device

        peft_cfg = PeftConfig.from_pretrained(lora_path)
        self.base_model_name = peft_cfg.base_model_name_or_path

        try:
            self.tokenizer = AutoTokenizer.from_pretrained(lora_path)
        except Exception:
            self.tokenizer = AutoTokenizer.from_pretrained(self.base_model_name)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        if device == "cuda":
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            base_model = AutoModelForCausalLM.from_pretrained(
                self.base_model_name,
                quantization_config=bnb_config,
                device_map="auto",
                output_hidden_states=True,
            )
        else:
            base_model = AutoModelForCausalLM.from_pretrained(
                self.base_model_name,
                torch_dtype=torch.float32,
                output_hidden_states=True,
            ).to(device)

        self.model = PeftModel.from_pretrained(base_model, lora_path, is_trainable=False)
        self.model.eval()

    def build_encoder_prompt(self, sparql: str) -> str:
        return f"""You are an expert in knowledge graphs.

Convert the SPARQL query into a clear natural language question.

### Instruction
Convert the SPARQL query into a clear natural language question.

### SPARQL Query
{sparql}

### Natural Text
"""

    @torch.no_grad()
    def encode_sparql(self, sparql_list: List[str], batch_size: int = 8, max_length: int = 768) -> np.ndarray:
        all_embeddings: List[torch.Tensor] = []
        model_device = next(self.model.parameters()).device

        for start in tqdm(range(0, len(sparql_list), batch_size), desc="Encoding SPARQL"):
            batch = sparql_list[start : start + batch_size]
            prompts = [self.build_encoder_prompt(s) for s in batch]

            toks = self.tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            )
            toks = {k: v.to(model_device) for k, v in toks.items()}

            outputs = self.model(**toks, output_hidden_states=True, return_dict=True)
            last_hidden = outputs.hidden_states[-1]  # (B, T, H)

            attn = toks["attention_mask"].unsqueeze(-1).float()  # (B, T, 1)
            pooled = (last_hidden * attn).sum(dim=1) / attn.sum(dim=1).clamp(min=1e-9)  # (B, H)
            pooled = l2_normalize(pooled)

            all_embeddings.append(pooled.cpu())

        return torch.cat(all_embeddings, dim=0).numpy()


# ============================================================
# Ground truth + Jaccard semantic relevance (strict)
# ============================================================

def answer_jaccard(a: List[str], b: List[str]) -> float:
    # IMPORTANT FIX:
    # If either side is empty, treat as 0 overlap.
    # We also removed the "empty-empty = 1.0" case.
    sa, sb = set(a), set(b)
    if len(sa) == 0 or len(sb) == 0:
        return 0.0
    return len(sa.intersection(sb)) / len(sa.union(sb))


def build_ground_truth_by_answer_jaccard(
    examples: List[Dict[str, Any]],
    threshold: float = 0.8,
    exclude_self: bool = True,
):
    n = len(examples)
    jaccard_matrix = np.zeros((n, n), dtype=np.float32)

    for i in tqdm(range(n), desc="Building answer-Jaccard matrix"):
        ai = examples[i]["answer"]
        for j in range(n):
            if exclude_self and i == j:
                continue
            aj = examples[j]["answer"]
            jaccard_matrix[i, j] = answer_jaccard(ai, aj)

    gt: Dict[int, List[int]] = {}
    for i in range(n):
        positives = [j for j in range(n) if jaccard_matrix[i, j] >= threshold]
        if exclude_self:
            positives = [j for j in positives if j != i]
        gt[i] = positives

    return gt, jaccard_matrix


# ============================================================
# Retrieval metrics
# ============================================================

def mask_self_matches(sim_matrix: np.ndarray) -> np.ndarray:
    sim = sim_matrix.copy()
    n, m = sim.shape
    if n == m:
        diag = np.arange(n)
        sim[diag, diag] = -1e9
    return sim


def get_ranked_indices(sim_matrix: np.ndarray) -> np.ndarray:
    return np.argsort(-sim_matrix, axis=1)


def recall_at_k(ranked_indices: np.ndarray, gt: Dict[int, List[int]], ks: List[int]) -> Dict[str, float]:
    ks = sorted(ks)
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]

    out: Dict[str, float] = {}
    for k in ks:
        hits = 0
        for i in valid_anchors:
            gt_set = set(gt[i])
            topk = ranked_indices[i, :k]
            if gt_set.intersection(topk):
                hits += 1
        out[f"Recall@{k}"] = hits / max(len(valid_anchors), 1)
    return out


def mrr_at_k(ranked_indices: np.ndarray, gt: Dict[int, List[int]], k: int) -> float:
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]
    rr = []

    for i in valid_anchors:
        gt_set = set(gt[i])
        topk = ranked_indices[i, :k]
        reciprocal_rank = 0.0
        for rank, idx in enumerate(topk, start=1):
            if idx in gt_set:
                reciprocal_rank = 1.0 / rank
                break
        rr.append(reciprocal_rank)

    return float(np.mean(rr)) if rr else 0.0


def dcg_at_k(relevances: List[float], k: int) -> float:
    rel = np.asarray(relevances[:k], dtype=np.float32)
    if rel.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, rel.size + 2))
    return float(np.sum(rel * discounts))


def ndcg_at_k(ranked_indices: np.ndarray, relevance_matrix: np.ndarray, gt: Dict[int, List[int]], k: int) -> float:
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]
    vals = []

    for i in valid_anchors:
        topk = ranked_indices[i, :k]
        rels = [float(relevance_matrix[i, j]) for j in topk]
        dcg = dcg_at_k(rels, k)

        # ideal DCG: best possible relevances among candidates excluding self
        ideal_rels = sorted(
            [float(relevance_matrix[i, j]) for j in range(relevance_matrix.shape[1]) if j != i],
            reverse=True,
        )
        idcg = dcg_at_k(ideal_rels, k)
        vals.append(0.0 if idcg == 0.0 else dcg / idcg)

    return float(np.mean(vals)) if vals else 0.0


def best_jaccard_at_k(ranked_indices: np.ndarray, jaccard_matrix: np.ndarray, gt: Dict[int, List[int]], k: int) -> float:
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]
    vals = []

    for i in valid_anchors:
        topk = ranked_indices[i, :k]
        vals.append(float(np.max(jaccard_matrix[i, topk])) if len(topk) > 0 else 0.0)

    return float(np.mean(vals)) if vals else 0.0


def mean_jaccard_at_k(ranked_indices: np.ndarray, jaccard_matrix: np.ndarray, gt: Dict[int, List[int]], k: int) -> float:
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]
    vals = []

    for i in valid_anchors:
        topk = ranked_indices[i, :k]
        vals.append(float(np.mean(jaccard_matrix[i, topk])) if len(topk) > 0 else 0.0)

    return float(np.mean(vals)) if vals else 0.0


def evaluate_ranking(sim_matrix: np.ndarray, gt: Dict[int, List[int]], jaccard_matrix: np.ndarray, ks: List[int]):
    sim_matrix = mask_self_matches(sim_matrix)
    ranked = get_ranked_indices(sim_matrix)

    results: Dict[str, float] = {}
    results.update(recall_at_k(ranked, gt, ks))

    max_k = max(ks)
    results[f"MRR@{max_k}"] = mrr_at_k(ranked, gt, max_k)

    for k in ks:
        results[f"nDCG@{k}"] = ndcg_at_k(ranked, jaccard_matrix, gt, k)
        results[f"BestJaccard@{k}"] = best_jaccard_at_k(ranked, jaccard_matrix, gt, k)
        results[f"MeanJaccard@{k}"] = mean_jaccard_at_k(ranked, jaccard_matrix, gt, k)

    results["NumAnchorsWithPositives"] = int(sum(1 for i in range(sim_matrix.shape[0]) if len(gt.get(i, [])) > 0))
    return results, ranked


def print_results(title: str, results: Dict[str, float]):
    print(f"\n=== {title} ===")
    for k, v in results.items():
        if k == "NumAnchorsWithPositives":
            print(f"{k}: {v}")
        else:
            print(f"{k}: {v:.4f}")


def print_retrieval_examples(
    title: str,
    ranked: np.ndarray,
    examples: List[Dict[str, Any]],
    jaccard_matrix: np.ndarray,
    num_examples: int = 5,
):
    print(f"\n===== Retrieval preview: {title} =====")
    shown = 0

    for i in range(len(examples)):
        topk = ranked[i, :5]
        if topk.size == 0:
            continue

        print("\n" + "=" * 80)
        print(f"ANCHOR #{i} | id={examples[i]['id']}")
        print(f"SPARQL:\n{examples[i]['sparql']}")
        print(f"GOLD QUESTION:\n{examples[i]['question']}")
        print(f"GOLD ANSWER: {examples[i]['answer']}")

        print("\nTop-5 retrieved SPARQL queries:")
        for rank, j in enumerate(topk, start=1):
            print(f"\n[Rank {rank}] corpus_idx={j}")
            print(f"Question: {examples[j]['question']}")
            print(f"SPARQL: {examples[j]['sparql'][:200]}...")
            print(f"Answer: {examples[j]['answer']}")
            print(f"Answer-Jaccard(anchor, retrieved) = {jaccard_matrix[i, j]:.4f}")

        print("\nSummary:")
        print(f"Top-1 Jaccard: {jaccard_matrix[i, topk[0]]:.4f}")
        print(f"Best Jaccard in Top-3: {np.max(jaccard_matrix[i, topk[:3]]):.4f}")
        print(f"Best Jaccard in Top-5: {np.max(jaccard_matrix[i, topk[:5]]):.4f}")

        shown += 1
        if shown >= num_examples:
            break


# ============================================================
# Finetune + RQ1
# ============================================================

def finetune_qwen_sparql2text():
    ensure_dir(OUTPUT_DIR)
    cleanup_memory()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_ds, val_ds = build_tokenized_train_val_datasets(tokenizer)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )

    model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False

    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    training_args = build_training_args()
    data_collator = SupervisedDataCollator(tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
    )

    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    save_json(trainer.state.log_history, os.path.join(OUTPUT_DIR, "log_history.json"))

    print(f"Saved fine-tuned LoRA model to: {OUTPUT_DIR}")


def run_rq1_retrieval_eval():
    cleanup_memory()

    test_examples = load_examples(os.path.join(DATA_DIR, "test.json"), USE_FIELD)
    print(f"Loaded {len(test_examples)} test examples (non-empty answers only)")

    gt, jaccard_matrix = build_ground_truth_by_answer_jaccard(
        test_examples,
        threshold=ANSWER_JACCARD_THRESHOLD,
        exclude_self=True,
    )

    num_with_pos = sum(1 for i in range(len(test_examples)) if len(gt.get(i, [])) > 0)
    print(f"Anchors with at least one positive: {num_with_pos}/{len(test_examples)}")

    sparql_list = [x["sparql"] for x in test_examples]

    qwen_encoder = QwenSPARQLEncoder(OUTPUT_DIR, device=DEVICE)

    # IMPORTANT FIX:
    # encode once; corpus and anchors are same list in this script
    emb = qwen_encoder.encode_sparql(
        sparql_list,
        batch_size=EMBED_BATCH_SIZE,
        max_length=MAX_SEQ_LENGTH,
    )

    sim_matrix = cosine_sim_matrix(emb, emb)

    results, ranked = evaluate_ranking(
        sim_matrix=sim_matrix,
        gt=gt,
        jaccard_matrix=jaccard_matrix,
        ks=TOPK,
    )

    print_results("Qwen fine-tuned SPARQL encoder (RQ1)", results)
    save_json(results, os.path.join(OUTPUT_DIR, "rq1_retrieval_metrics.json"))

    print_retrieval_examples(
        title="Qwen fine-tuned SPARQL encoder (RQ1)",
        ranked=ranked,
        examples=test_examples,
        jaccard_matrix=jaccard_matrix,
        num_examples=5,
    )


def main():
    set_seed(SEED)

    print("Step 1/2: Fine-tuning Qwen on SPARQL -> natural text...")
    finetune_qwen_sparql2text()

    print("\nStep 2/2: Evaluating RQ1 with fine-tuned Qwen as SPARQL encoder...")
    run_rq1_retrieval_eval()


if __name__ == "__main__":
    main()

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql-emb/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Step 1/2: Fine-tuning Qwen on SPARQL -> natural text...


Loading weights: 100%|██████████| 339/339 [00:05<00:00, 62.79it/s, Materializing param=model.norm.weight]                               


trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


Step,Training Loss,Validation Loss
100,2.781243,2.782962
200,2.626988,2.668298
300,2.590298,2.601045
400,2.494855,2.539181
500,2.633517,2.480853
600,2.480202,2.434632
700,2.435160,2.396223
800,2.229263,2.369055
900,2.301446,2.334085
1000,2.084562,2.314450


Saved fine-tuned LoRA model to: ./qwen_sparql2text_encoder_rq1

Step 2/2: Evaluating RQ1 with fine-tuned Qwen as SPARQL encoder...
Loaded 3052 test examples (non-empty answers only)


Building answer-Jaccard matrix: 100%|██████████| 3052/3052 [01:57<00:00, 26.04it/s]


Anchors with at least one positive: 562/3052


The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Encoding SPARQL: 100%|██████████| 382/382 [04:42<00:00,  1.35it/s]



=== Qwen fine-tuned SPARQL encoder (RQ1) ===
Recall@1: 0.2420
Recall@5: 0.3897
Recall@10: 0.4964
MRR@10: 0.3056
nDCG@1: 0.2452
BestJaccard@1: 0.2448
MeanJaccard@1: 0.2448
nDCG@5: 0.2126
BestJaccard@5: 0.3913
MeanJaccard@5: 0.1008
nDCG@10: 0.2119
BestJaccard@10: 0.4996
MeanJaccard@10: 0.0727
NumAnchorsWithPositives: 562

===== Retrieval preview: Qwen fine-tuned SPARQL encoder (RQ1) =====

ANCHOR #0 | id=0
SPARQL:
SELECT ?answer WHERE { ?statement1 <http://www.w3.org/1999/02/22-rdf-syntax-ns#subject> <http://wikidata.dbpedia.org/resource/Q16538> . ?statement1 <http://www.w3.org/1999/02/22-rdf-syntax-ns#predicate> <http://www.wikidata.org/entity/P725>. ?statement1 <http://www.w3.org/1999/02/22-rdf-syntax-ns#object> ?answer . ?statement2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#subject> ?answer. ?statement2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#predicate> <http://www.wikidata.org/entity/P106> . ?statement2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#object> <http://wikidata.dbped